# Lesson 86 · Sampling strategies from scratch — give the AI a little free will: temperature / top-k / top-p in pure NumPy

**Learning goals**
- Understand why greedy decoding tends to repeat itself
- Implement temperature scaling, top-k and top-p (nucleus) sampling
- Compare the distribution shapes of the three strategies side by side
- Cross-check against aurora.llm.sample

> Temperature = how sharp the logits are ($T\to 0$ is close to greedy, large $T$ is more random); top-p picks an adaptive nucleus of candidates.

← **Previous lesson** · [L85 · A KV cache from scratch](L85_kv_cache.ipynb)

> Last lesson covered **a KV cache from scratch**: updating the key/value cache, O(seq²) → O(seq) per step.
> This lesson explores **sampling strategies from scratch**.

## Why does an AI sometimes get more and more repetitive?

Show GPT "The sky is" and greedy decoding picks "blue" — because "blue" has the highest probability. But at the next step, given "The sky is blue", it may pick "blue" again… and so the sentence rolls on and on until it gets stuck in a loop.

**The root of the problem**: greedy decoding picks only the single most probable token at every step. The path is the safest one, but it is also the one most likely to circle around a local optimum until it talks itself into repetition.

**The fix**: don't squeeze the probability distribution too hard — leave it a little "breathing room" so low-probability tokens also get a chance to be picked. Three common strategies:

```
Temperature T: logits ÷ T
  T < 1 → sharper distribution, more conservative   (T=0 = greedy)
  T > 1 → flatter distribution, more random         (T=∞ = uniform random)

Top-k sampling: keep only the k most probable tokens, set the rest to -∞
  k=1  → greedy; k=vocab_size → the full distribution
  problem: k is fixed, no matter how sharp the distribution is

Top-p / nucleus sampling (Holtzman 2020): keep the smallest set of tokens whose cumulative probability ≥ p
  sharp distribution (one token dominates) → tiny nucleus (just 1–2 tokens)
  flat distribution (choices are even)     → large nucleus (many candidates)
```

**Recipe (building the top-p nucleus)**:
1. Sort: order all tokens by prob, largest first
2. Accumulate: `cumsum = [p₁, p₁+p₂, ...]`
3. Truncate: find the first position where cumsum ≥ p → those tokens form the nucleus
4. Renormalize over the nucleus and sample

This section implements `top_k_sample` and `top_p_sample` in pure NumPy and checks them against `aurora.llm.sample`.

## ⚠️ Common misconceptions

> Don't think sampling makes the model "smarter" or "more accurate". None of these three strategies changes the probabilities `probs` the model gives, and none conjures up new information — they only decide **how to pick one token from those probabilities**: greedy always picks the top one (prone to repetition), while temperature/top-k/top-p allow picks from the long tail (more variety). They turn the "determinism ↔ randomness" knob, not a "quality" knob.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
def softmax(logits, temperature=1.0):
    """Numerically stable softmax with temperature scaling."""
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits / max(float(temperature), 1e-8)
    logits -= logits.max()
    exp = np.exp(logits)
    return exp / exp.sum()

# Test logit distribution
VOCAB = 20
logits = np.random.randn(VOCAB) * 2   # simulate a sharp distribution

print('Effect of temperature on the distribution:')
for T in [0.5, 1.0, 2.0]:
    probs = softmax(logits, T)
    entropy = -np.sum(probs * np.log(probs + 1e-30))
    print(f'  T={T}: max_prob={probs.max():.3f}, entropy={entropy:.2f}')

## ✏️ Task 1: top-k sampling

**Signature**:
```python
def top_k_sample(logits, k=10, temperature=1.0) -> int:
    # returns: the sampled token index (an integer)
```

**Three steps to implement it**:

| Step | Action | Code hint |
|---|---|---|
| 1 | Find the top-k indices | `np.argpartition(logits, -k)[-k:]` |
| 2 | Set the other tokens to -inf | a new array `masked = -inf * ones`, then put the original logits back at the top-k positions |
| 3 | softmax + sample | `softmax(masked, temperature)` → `np.random.choice(vocab, p=probs)` |

**Acceptance criteria**:
- the return value is an integer inside the top-k set
- `k=1` is equivalent to greedy decoding (at temperature 1)

In [ ]:
def top_k_sample(logits, k=10, temperature=1.0):
    """Sample from the k tokens with the largest logits."""
    logits = np.asarray(logits, dtype=np.float64)
    k = max(1, min(k, len(logits)))
    # ✏️ TODO: find the top-k indices, set the rest to -inf, softmax, sample
    raise NotImplementedError("TODO")

## ✏️ Task 2: top-p (nucleus) sampling

**Signature**:
```python
def top_p_sample(logits, p=0.9, temperature=1.0) -> int:
    # returns: the sampled token index (an integer)
```

**Four steps to implement it**:

| Step | Action | Code hint |
|---|---|---|
| 1 | softmax → probs | `probs = softmax(logits, temperature)` |
| 2 | Sort by prob, descending | `sorted_idx = np.argsort(probs)[::-1]` |
| 3 | Cumulative probability, find the cutoff | `cumsum = np.cumsum(sorted_probs)` → `np.searchsorted(cumsum, p) + 1` |
| 4 | Renormalize within the nucleus and sample | `nucleus = sorted_idx[:cutoff]`; `np.random.choice` over the nucleus probs |

**Acceptance criteria**:
- the return value is an integer inside the nucleus set
- `p=1.0` covers the whole vocabulary (equivalent to plain temperature sampling)

In [ ]:
def top_p_sample(logits, p=0.9, temperature=1.0):
    """Sample from the smallest set of tokens whose cumulative probability is >= p."""
    logits = np.asarray(logits, dtype=np.float64)
    probs = softmax(logits, temperature)
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    # ✏️ TODO: take the cumulative sum and find the first position >= p
    raise NotImplementedError("TODO")

In [ ]:
try:
    # Visualize the sampling distributions of four strategies
    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    probs_base = softmax(logits, 1.0)
    axes = axes.flatten()

    strategies = [
        ('Greedy', [np.argmax(logits)]),
        ('Temperature T=0.5', [top_k_sample(logits, k=VOCAB, temperature=0.5) for _ in range(500)]),
        ('Top-k k=5', [top_k_sample(logits, k=5) for _ in range(500)]),
        ('Top-p p=0.9', [top_p_sample(logits, p=0.9) for _ in range(500)]),
    ]

    for ax, (name, samples) in zip(axes, strategies):
        counts = np.bincount(samples, minlength=VOCAB) / max(len(samples), 1)
        ax.bar(range(VOCAB), counts, alpha=0.7, label='sample freq')
        ax.plot(range(VOCAB), probs_base, 'r--', alpha=0.5, label='base prob')
        ax.set_title(name, fontsize=10)
        ax.set_xlabel('Token ID')
        ax.set_ylabel('Frequency')
        ax.legend(fontsize=8)

    plt.suptitle('Four sampling strategies compared (vocab_size=20)', fontsize=12)
    plt.tight_layout()
    plt.show()
    print('✅ sampling-distribution plot done')
except (NotImplementedError, TypeError):
    print('⬜ Implement top_k_sample / top_p_sample first, then run the visualization')


In [ ]:
try:
    # Test that top_k_sample returns a value inside the top-k set
    test_logits = np.random.randn(50)
    k = 10
    result_topk = top_k_sample(test_logits, k=k)
    assert result_topk is not None, "⬜ not implemented yet"
    assert isinstance(result_topk, (int, np.integer)), f"should return an integer index, got {type(result_topk)}"
    top_k_indices = set(np.argpartition(test_logits, -k)[-k:].tolist())
    assert result_topk in top_k_indices, f"return value {result_topk} is not in the top-{k} set"
    print(f'✅ top_k_sample returned: {result_topk} (inside the top-{k} set)')

    # Test that top_p_sample returns a valid index that lands inside the nucleus
    result_topp = top_p_sample(test_logits, p=0.9)
    assert result_topp is not None, "⬜ not implemented yet"
    assert isinstance(result_topp, (int, np.integer)), f"should return an integer index, got {type(result_topp)}"
    assert 0 <= result_topp < len(test_logits), f"return value {result_topp} is out of range"

    probs = np.exp(test_logits - test_logits.max())
    probs /= probs.sum()
    sorted_idx = np.argsort(probs)[::-1]
    cumsum = np.cumsum(probs[sorted_idx])
    cutoff = np.searchsorted(cumsum, 0.9) + 1
    nucleus = set(sorted_idx[:cutoff].tolist())
    assert result_topp in nucleus, f"return value {result_topp} is not in the nucleus {nucleus}."
    print(f'✅ top_p_sample returned: {result_topp} (inside the nucleus)')

    # Compare with aurora.llm
    from aurora.llm import top_k_sample as ref_topk
    ref_result = ref_topk(test_logits, k=k)
    ref_top_k_indices = set(np.argpartition(test_logits, -k)[-k:].tolist())
    assert ref_result in ref_top_k_indices, f"aurora.llm reference returned {ref_result}, which is not in the top-{k} set"
    print('✅ sampling strategies verified')
except (NotImplementedError, TypeError):
    print('⬜ not implemented yet')

## Wrap-up

| Strategy | Parameter | Effect |
|---|---|---|
| Greedy | none | deterministic, prone to repetition |
| Temperature | T | T<1 sharper, T>1 more random |
| Top-k | k | fixed number of candidates |
| Top-p | p | adaptive candidate set; high confidence → small nucleus |

Real LLMs usually combine them: `top_p_sample(logits, p=0.9, temperature=0.8)`.

Next up, L87: first the principle of 8-bit quantization, then hooking it up to Hugging Face for real inference.

## ✏️ Closed-book derivation check cell — deriving the top-p nucleus

**Rule: collapse every cell above and do the derivation below from memory alone.**

**Problem**: given logits = [2.0, 1.0, 0.5, 0.1, -1.0] (vocabulary size 5) and p = 0.9

1. Write out the softmax steps (including the numerical-stability trick)
2. Sort by probability, largest first, and fill in the cumulative-probability table:

| Rank | token index | prob | cumsum |
|------|------------|------|--------|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

3. Which token indices form the nucleus for p=0.9?

(Write your derivation here...)

In [ ]:
# Check: hand calculation agrees with aurora.llm.sample
import numpy as np, sys; sys.path.insert(0, 'src')
from aurora.llm.sample import top_p_sample

logits = np.array([2.0, 1.0, 0.5, 0.1, -1.0])
p = 0.9

# Rebuild the nucleus by hand
probs = np.exp(logits - logits.max())
probs /= probs.sum()
sorted_idx = np.argsort(probs)[::-1]
sorted_probs = probs[sorted_idx]
cumsum = np.cumsum(sorted_probs)

# nucleus = every token with cumsum ≤ p + the first token that goes past p
cutoff = np.searchsorted(cumsum, p) + 1
nucleus_tokens = set(sorted_idx[:cutoff].tolist())

print("Nucleus token indices:", sorted(nucleus_tokens))
print("Cumulative probs:", [f"{c:.3f}" for c in cumsum[:cutoff]])

# Sample many times and check every result lands inside the nucleus
np.random.seed(99)
for _ in range(200):
    sampled = top_p_sample(logits, p=p)
    assert sampled in nucleus_tokens, f"sample {sampled} is not in the nucleus {nucleus_tokens}"
print(f"✅ top-p nucleus verified (all 200 samples inside the nucleus)")

In [ ]:
# ✏️ Self-review for this lesson
l86_review = {
    "greedy_problem":            None,  # understand greedy decoding's repetition problem: deterministic → local optimum? True/False
    "temperature_intuition":     None,  # understand temperature: T<1 sharper / T>1 flatter, T=0 = greedy? True/False
    "top_k_impl":                None,  # top_k_sample correct: return value inside the top-k set? True/False
    "top_p_impl":                None,  # top_p_sample correct: nucleus is the smallest set covering ≥ p? True/False
    "whiteboard_passed":         None,  # whiteboard challenge (top-p nucleus derivation) done? True/False
}

unfilled = [k for k, v in l86_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l86_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L86 fully cleared! On to L87: quantization and local inference')

---

→ **Next lesson** · [L87 · Quantization and local inference](L87_local_inference.ipynb)

> Next lesson covers **quantization and local inference**: how 8-bit quantization works, and lightweight inference with Hugging Face.